# Solar Filament Instance Segmentation

This notebook trains an instance segmentation pipeline for solar filaments in full-disk H-alpha observations. A pretrained detector identifies individual filament candidates. A separate mask refiner reconstructs each candidate's outline from its local image context. Calibration selects the model output and filtering settings used for validation and test prediction.

## Data and execution

| Input | Use |
|---|---|
| 707 training images | Model fitting, calibration and held-out validation |
| Competition training annotation JSON | Filament mask supervision and accuracy measurement |
| 180 test images | Final prediction and submission generation |

Attach these official competition inputs to a Kaggle notebook. Enable a GPU and Internet access to install the dependencies and obtain pretrained weights. Run the cells in order. The setup checks the dataset, builds the partitions, and tests model execution and mask encoding before training.

## Method

1. Keep observations from nearby dates in the same group and create separate training, calibration and validation partitions.
2. Train a COCO-pretrained YOLO11m segmentation detector using full-disk images resized to 1536 pixels.
3. Build instance crops from the original annotated masks, together with dark background regions and false detector proposals.
4. Train a ResNet18 mask refiner on 384-pixel crops containing grayscale intensity, CLAHE contrast and a candidate seed box.
5. Select checkpoints and filtering settings on the calibration partition, then measure the frozen pipeline on held-out images.
6. Predict filament masks for the complete test set, encode them as COCO RLE, and check the submission CSV.

The configuration cell defines the image sizes, folds, batch sizes and training duration. The default configuration runs one fold, up to 30 detector epochs and up to 15 refiner epochs. Timing estimates are calculated from the running session. Metric tables and segmentation previews are generated when the evaluation cells execute.


In [ ]:
# Dependencies: keep Kaggle's matching torch/torchvision builds.
import os, subprocess, sys
os.environ['WANDB_DISABLED'] = 'true'
os.environ['YOLO_AUTOINSTALL'] = 'false'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'ultralytics==8.3.221', 'pycocotools==2.0.10'], check=True)


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, shutil, time, zipfile
from IPython.display import FileLink, display

PROJECT = Path('/kaggle/working/filament_project_v5')
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCES = {'validate_submission.py': '"""Decode every test mask, validate IDs and enforce disjoint instances."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom pycocotools import mask as coco\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--submission\',required=True)\n    parser.add_argument(\'--images\',required=True)\n    args=parser.parse_args()\n    table=pd.read_csv(args.submission,dtype=str,keep_default_na=False)\n    if list(table.columns)!=[\'filament_id\',\'segmentation_rle\']:\n        raise ValueError(\'Incorrect submission columns\')\n    if table.filament_id.duplicated().any():raise ValueError(\'Duplicate filament IDs\')\n    if not table.filament_id.str.fullmatch(r\'[0-9]{14}[A-Za-z]{2}_[1-9][0-9]*\').all():raise ValueError(\'Invalid filament IDs\')\n    names={p.stem for p in Path(args.images).iterdir() if p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\'}}\n    if not names:raise ValueError(\'No test images found\')\n    table[\'image_id\']=table.filament_id.str.rsplit(\'_\',n=1).str[0]\n    if not set(table.image_id)<=names:raise ValueError(\'Submission contains unknown image IDs\')\n    areas=[]\n    for name,group in table.groupby(\'image_id\',sort=False):\n        occupied=np.zeros((2048,2048),bool)\n        for row in group.itertuples(index=False):\n            rle={\'size\':[2048,2048],\'counts\':row.segmentation_rle.encode(\'ascii\')}\n            mask=coco.decode(rle).astype(bool)\n            area=int(mask.sum())\n            if mask.shape!=(2048,2048) or area==0:raise ValueError(f\'Empty or invalid mask: {row.filament_id}\')\n            if np.any(occupied&mask):raise ValueError(f\'Overlapping predicted instances: {name}\')\n            occupied|=mask;areas.append(area)\n    report={\'status\':\'passed\',\'test_images\':len(names),\'predicted_instances\':len(table),\n            \'images_with_predictions\':int(table.image_id.nunique()),\n            \'images_without_predictions\':sorted(names-set(table.image_id)),\n            \'minimum_instance_area\':min(areas) if areas else None,\n            \'all_masks_decoded\':True,\'prediction_instances_disjoint\':True}\n    Path(args.submission).with_suffix(\'.checks.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'prepare_data.py': '"""Unify split ZIP uploads or extracted competition folders without mixing train/test."""\nimport argparse,hashlib,json,shutil,zipfile\nfrom pathlib import Path\nimport pandas as pd\nfrom filament.data import AnnotationIndex,make_folds\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument("--input",required=True);p.add_argument("--output",required=True);p.add_argument("--folds",default="reports/folds.csv");args=p.parse_args()\n    source=Path(args.input).resolve();out=Path(args.output).resolve()\n    if source==out or source in out.parents:raise ValueError("Output must be outside the input tree")\n    candidates=list(source.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))\n    if not candidates:raise FileNotFoundError("Attach the training JSON separately or use an extracted competition dataset")\n    if len({hashlib.sha256(p.read_bytes()).hexdigest() for p in candidates})!=1:raise ValueError("Conflicting annotation JSON copies")\n    out.mkdir(parents=True,exist_ok=True);annotations=out/candidates[0].name;shutil.copyfile(candidates[0],annotations)\n    index=AnnotationIndex(annotations);train_names=set(index.filenames)\n    for split in ["train_images","test_images"]:(out/split).mkdir(exist_ok=True)\n    def place(name,split,raw=None,path=None):\n        dest=out/split/name\n        if dest.exists():\n            incoming=raw if raw is not None else path.read_bytes()\n            if hashlib.sha256(dest.read_bytes()).digest()!=hashlib.sha256(incoming).digest():raise ValueError(f"Conflicting image: {name}")\n        elif raw is not None:dest.write_bytes(raw)\n        else:dest.symlink_to(path.resolve())\n    for path in source.rglob("*"):\n        if not path.is_file() or path.suffix.lower() not in {".jpg",".jpeg",".png"}:continue\n        if path.name in train_names:place(path.name,"train_images",path=path)\n        elif "test_images" in path.parts:place(path.name,"test_images",path=path)\n    for archive in sorted(source.rglob("*.zip")):\n        with zipfile.ZipFile(archive) as z:\n            for item in z.infolist():\n                name=Path(item.filename).name\n                if item.is_dir() or Path(name).suffix.lower() not in {".jpg",".jpeg",".png"}:continue\n                if name in train_names:split="train_images"\n                elif "test_images" in Path(item.filename).parts or archive.name.lower()=="test.zip":split="test_images"\n                else:continue\n                place(name,split,raw=z.read(item))\n    missing=train_names-{p.name for p in (out/"train_images").glob("*")}\n    if missing:raise ValueError(f"Missing {len(missing)} training images; examples: {sorted(missing)[:5]}")\n    test_names={p.name for p in (out/"test_images").glob("*")}\n    if train_names&test_names:raise ValueError("Training/test filename overlap")\n    folds=Path(args.folds);folds.parent.mkdir(parents=True,exist_ok=True);pd.DataFrame(make_folds(index)).to_csv(folds,index=False)\n    print(json.dumps({"train_images":len(train_names),"test_images":len(test_names),"annotations":str(annotations),"train_directory":str(out/"train_images"),"test_directory":str(out/"test_images")},indent=2))\n\nif __name__=="__main__":main()\n', 'geometry.py': '"""Geometry and leakage checks, kept separate for CPU testing."""\nimport numpy as np\n\n\ndef context_crop(box, width, height, padding=1.5, minimum=48):\n    x1, y1, x2, y2 = map(float, box)\n    side = min(max((x2-x1)*padding, (y2-y1)*padding, minimum), max(width, height))\n    cx, cy = (x1+x2)/2, (y1+y2)/2\n    xa = max(0, min(int(np.floor(cx-side/2)), max(0, width-int(np.ceil(side)))))\n    ya = max(0, min(int(np.floor(cy-side/2)), max(0, height-int(np.ceil(side)))))\n    xb, yb = min(width, max(xa+1, int(np.ceil(xa+side)))), min(height, max(ya+1, int(np.ceil(ya+side))))\n    return xa, ya, xb, yb\n\n\ndef box_iou(box, boxes):\n    boxes = np.asarray(boxes, float).reshape(-1, 4)\n    if not len(boxes): return np.zeros(0)\n    low = np.maximum(np.asarray(box)[:2], boxes[:, :2]); high = np.minimum(np.asarray(box)[2:], boxes[:, 2:])\n    intersection = np.prod(np.maximum(0, high-low), axis=1)\n    a = max(0, box[2]-box[0])*max(0, box[3]-box[1])\n    b = np.prod(np.maximum(0, boxes[:, 2:]-boxes[:, :2]), axis=1)\n    return intersection / np.maximum(a+b-intersection, 1e-9)\n\n\ndef assert_partitions(partitions, groups):\n    seen_names, seen_groups = set(), set()\n    for key in (\'train\', \'calibration\', \'validation\'):\n        names = set(partitions[key]); current = {groups[n] for n in names}\n        if names & seen_names or current & seen_groups:\n            raise ValueError(\'Physical images or temporal groups overlap between partitions\')\n        if not names: raise ValueError(\'Empty partition: \'+key)\n        seen_names |= names; seen_groups |= current\n\n\ndef ordered_disjoint(patches, shape, minimum):\n    """Highest score owns overlap; fragments of an instance stay in one mask."""\n    occupied = np.zeros(shape, bool); kept = []\n    for record in sorted(patches, key=lambda r: -r[\'score\']):\n        x1, y1, x2, y2 = record[\'crop\']; mask = np.asarray(record[\'mask\'], bool)\n        if mask.shape != (y2-y1, x2-x1): raise ValueError(\'Crop/mask shape mismatch\')\n        local = mask & ~occupied[y1:y2, x1:x2]\n        if local.sum() < minimum: continue\n        occupied[y1:y2, x1:x2] |= local\n        kept.append({**record, \'mask\': local})\n    return kept\n', 'predict_test.py': '"""Test inference uses test images and frozen settings; no annotation input."""\nimport argparse,importlib.metadata,json,time\nfrom pathlib import Path\nimport pandas as pd\nfrom pipeline import Pipeline,prepare_candidates,select_prepared\nfrom cached_evaluation import sha,atomic_json\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'experiment\',\'images\',\'out\'):p.add_argument(\'--\'+name,required=True)\n    args=p.parse_args();experiment=Path(args.experiment);settings=json.loads((experiment/\'selected_settings.json\').read_text())\n    detector=experiment/\'detector_selected.pt\';refiner=experiment/\'refiner_best.pt\' if settings[\'mode\']==\'refiner\' else None\n    descriptor={\'detector\':sha(detector),\'refiner\':sha(refiner) if refiner else None,\'runtime_versions\':{p:importlib.metadata.version(p) for p in (\'torch\',\'torchvision\',\'ultralytics\',\'numpy\',\'pycocotools\')},\'settings\':settings,\'version\':\'detref-v5-1\',\'source_sha256\':{f:sha(Path(__file__).parent/f) for f in (\'pipeline.py\',\'patches.py\',\'geometry.py\',\'refiner.py\')}}\n    pipeline=None;rows=[];empty=[];names=sorted(p for p in Path(args.images).iterdir() if p.suffix.lower() in {\'.jpeg\',\'.jpg\',\'.png\'})\n    if len(names)!=180:raise ValueError(f\'Expected 180 competition test images, found {len(names)}\')\n    cache_dir=experiment/\'test_cache\';cache_dir.mkdir(exist_ok=True);start=time.monotonic()\n    for i,path in enumerate(names):\n        fingerprint={**descriptor,\'image_sha256\':sha(path)};cache=cache_dir/(path.stem+\'.json\');data=json.loads(cache.read_text()) if cache.exists() else None\n        if data is None or data.get(\'fingerprint\')!=fingerprint:\n            if pipeline is None:pipeline=Pipeline(detector,refiner,settings[\'imgsz\'],settings[\'tta\'])\n            candidates=pipeline.candidates(path);prepared=prepare_candidates(candidates,settings[\'mode\'],settings[\'mask_threshold\'])\n            rles=select_prepared(prepared,settings);data={\'fingerprint\':fingerprint,\'rles\':rles};atomic_json(cache,data)\n        rles=data[\'rles\'];rows.extend({\'filament_id\':f\'{path.stem}_{j}\',\'segmentation_rle\':r[\'counts\']} for j,r in enumerate(rles,1))\n        if not rles:empty.append(path.name)\n        if (i+1)%20==0:print(f\'Predicted {i+1}/{len(names)} | elapsed {(time.monotonic()-start)/60:.1f} min\',flush=True)\n    out=Path(args.out);pd.DataFrame(rows,columns=[\'filament_id\',\'segmentation_rle\']).to_csv(out,index=False)\n    summary={\'images\':len(names),\'predicted_instances\':len(rows),\'images_with_no_predictions\':empty,\'seconds\':time.monotonic()-start,\'settings\':settings,\'annotations_used\':False}\n    out.with_suffix(\'.metrics.json\').write_text(json.dumps(summary,indent=2));print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':main()\n', 'metric_core.py': '"""Pooled instance metrics and temporal-group confidence intervals."""\nimport numpy as np\n\ndef counts(iou):\n    iou = np.asarray(iou, dtype=float)\n    hit, overlap = (iou > 0.5, iou > 0)\n    return np.array([hit.sum(), (hit.sum(0) == 0).sum(), (hit.sum(1) == 0).sum(), iou[hit].sum(), (overlap.sum(1) > 1).sum(), (overlap.sum(0) > 1).sum()], float)\n\ndef report(total):\n    tp, fp, fn, summed, split, merge = map(float, total)\n    denom = tp + 0.5 * fp + 0.5 * fn\n    return dict(tp=int(tp), fp=int(fp), fn=int(fn), sum_iou=summed, split_gt=int(split), merged_predictions=int(merge), pq=summed / denom if denom else 0.0, sq=summed / tp if tp else 0.0, rq=tp / denom if denom else 0.0)\n\ndef grouped_interval(baseline_counts, candidate_counts, groups, seed=2026, draws=1000):\n    """Paired temporal-group bootstrap of pooled PQ differences."""\n    base, candidate = (np.asarray(baseline_counts), np.asarray(candidate_counts))\n    groups = np.asarray(groups)\n    unique = np.unique(groups)\n    if len(unique) < 2:\n        return None\n    a = np.stack([base[groups == group].sum(0) for group in unique])\n    b = np.stack([candidate[groups == group].sum(0) for group in unique])\n    rng = np.random.default_rng(seed)\n    delta = []\n    for _ in range(draws):\n        sample = rng.integers(0, len(unique), len(unique))\n        delta.append(report(b[sample].sum(0))[\'pq\'] - report(a[sample].sum(0))[\'pq\'])\n    low, high = np.quantile(delta, [0.025, 0.975])\n    return {\'lower\': float(low), \'upper\': float(high), \'draws\': draws, \'groups\': len(unique)}\n', 'refiner.py': '"""ResNet18 encoder, GroupNorm decoder, mask and foreground-presence heads."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import resnet18,ResNet18_Weights\n\n\nclass Block(nn.Sequential):\n    def __init__(self,inp,out):\n        super().__init__(nn.Conv2d(inp,out,3,padding=1,bias=False),nn.GroupNorm(8,out),nn.SiLU(),\n                         nn.Conv2d(out,out,3,padding=1,bias=False),nn.GroupNorm(8,out),nn.SiLU())\n\n\nclass Refiner(nn.Module):\n    def __init__(self,pretrained=False,encoder_weights=None):\n        super().__init__()\n        backbone=resnet18(weights=ResNet18_Weights.DEFAULT if pretrained and not encoder_weights else None)\n        if encoder_weights:backbone.load_state_dict(torch.load(encoder_weights,map_location=\'cpu\',weights_only=True))\n        self.stem=nn.Sequential(backbone.conv1,backbone.bn1,backbone.relu)\n        self.pool=backbone.maxpool;self.e1=backbone.layer1;self.e2=backbone.layer2;self.e3=backbone.layer3;self.e4=backbone.layer4\n        self.d3=Block(512+256,256);self.d2=Block(256+128,128);self.d1=Block(128+64,64);self.d0=Block(64+64,32)\n        self.mask=nn.Sequential(Block(32,32),nn.Conv2d(32,1,1))\n        self.presence=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(512,1))\n        self.register_buffer(\'mean\',torch.tensor([.485,.456,.406])[None,:,None,None])\n        self.register_buffer(\'std\',torch.tensor([.229,.224,.225])[None,:,None,None])\n    def train(self,mode=True):\n        super().train(mode)\n        # Frozen encoder running statistics support small instance batches.\n        for module in self.modules():\n            if isinstance(module,nn.BatchNorm2d):module.eval()\n        return self\n    def forward(self,x):\n        size=x.shape[-2:];x=(x-self.mean)/self.std;s=self.stem(x)\n        a=self.e1(self.pool(s));b=self.e2(a);c=self.e3(b);d=self.e4(c)\n        up=lambda x,y:F.interpolate(x,size=y.shape[-2:],mode=\'bilinear\',align_corners=False)\n        y=self.d3(torch.cat([up(d,c),c],1));y=self.d2(torch.cat([up(y,b),b],1))\n        y=self.d1(torch.cat([up(y,a),a],1));y=self.d0(torch.cat([up(y,s),s],1))\n        return F.interpolate(self.mask(y),size=size,mode=\'bilinear\',align_corners=False),self.presence(d).flatten()\n\n\ndef objective(logits,presence,truth,valid,positive):\n    valid=valid.float();prob=logits.sigmoid()\n    dilated=F.max_pool2d(truth,3,1,1);eroded=-F.max_pool2d(-truth,3,1,1)\n    weight=valid*(1+3*(dilated-eroded).clamp(0,1))\n    bce=(F.binary_cross_entropy_with_logits(logits,truth,reduction=\'none\')*weight).sum()/weight.sum().clamp_min(1)\n    dims=(1,2,3);intersection=(prob*truth*valid).sum(dims)\n    dice=1-(2*intersection+1)/((prob*valid).sum(dims)+(truth*valid).sum(dims)+1)\n    positive_dice=dice[positive>0].mean() if (positive>0).any() else logits.sum()*0\n    detection=F.binary_cross_entropy_with_logits(presence,positive.float())\n    return bce+positive_dice+.3*detection\n', 'smoke_test.py': '"""One forward/backward pass and round-trip encoding before expensive training."""\nimport argparse,json\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\nfrom pycocotools import mask as coco\nfrom filament.data import encode_mask\nfrom refiner import Refiner,objective\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--detector-weights\',required=True);p.add_argument(\'--encoder-weights\');p.add_argument(\'--image\',required=True);p.add_argument(\'--size\',type=int,default=384);args=p.parse_args()\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a GPU.\')\n    model=Refiner(pretrained=True,encoder_weights=args.encoder_weights).cuda().train()\n    x=torch.rand(2,3,args.size,args.size,device=\'cuda\');y=torch.zeros(2,1,args.size,args.size,device=\'cuda\');y[0,0,80:180,100:110]=1\n    valid=torch.ones_like(y);positive=torch.tensor([1.,0.],device=\'cuda\')\n    logits,presence=model(x);loss=objective(logits,presence,y,valid,positive)\n    assert logits.shape==y.shape and presence.shape==(2,) and torch.isfinite(loss)\n    loss.backward();gradients=[p.grad for p in model.parameters() if p.grad is not None];assert gradients and all(torch.isfinite(g).all() for g in gradients)\n    mask=np.zeros((2048,2048),np.uint8);mask[1:20,2040:2046]=1\n    assert np.array_equal(coco.decode(encode_mask(mask)),mask)\n    del model,x,y,valid,positive,logits,presence,loss;torch.cuda.empty_cache()\n    result=YOLO(args.detector_weights).predict(args.image,imgsz=512,device=0,retina_masks=True,verbose=False)[0]\n    if result.masks is not None:assert tuple(result.masks.data.shape[-2:])==tuple(result.orig_shape)\n    print(json.dumps({\'smoke_test\':\'passed\',\'gpu\':torch.cuda.get_device_name(0),\'refiner_backward\':True,\'rle_round_trip\':True,\'detector_api\':True}))\n\nif __name__==\'__main__\':main()\n', 'cached_evaluation.py': '"""One IoU calculation per method/image; threshold comparisons reuse matrices."""\nimport hashlib,importlib.metadata,json,time\nfrom pathlib import Path\nimport numpy as np\nfrom filament.metrics import pairwise_iou\nfrom metric_core import counts,report\nfrom pipeline import Pipeline,prepare_candidates\n\n\ndef sha(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'):digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path,data):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);temp=path.with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(data));temp.replace(path)\n\n\ndef method_key(settings):\n    return \'detector\' if settings[\'mode\']==\'detector\' else f"refiner_{settings[\'mask_threshold\']}"\n\n\ndef collect(names,images,index,annotations,detector,refiner,imgsz,out,tta=True):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    descriptor={\'detector_sha256\':sha(detector),\'refiner_sha256\':sha(refiner) if refiner else None,\n        \'runtime_versions\':{p:importlib.metadata.version(p) for p in (\'torch\',\'torchvision\',\'ultralytics\',\'numpy\',\'pycocotools\')},\n        \'annotations_sha256\':sha(annotations),\'imgsz\':imgsz,\'tta\':tta,\'version\':\'detref-v5-1\',\n        \'source_sha256\':{f:sha(Path(__file__).parent/f) for f in (\'pipeline.py\',\'patches.py\',\'geometry.py\',\'refiner.py\',\'cached_evaluation.py\')}}\n    fingerprint=hashlib.sha256(json.dumps(descriptor,sort_keys=True).encode()).hexdigest()\n    pipeline=None;rows=[];started=time.monotonic()\n    for i,name in enumerate(names):\n        path=Path(images)/name;key=hashlib.sha256((fingerprint+sha(path)).encode()).hexdigest();cache=out/(Path(name).stem+\'.json\')\n        row=json.loads(cache.read_text()) if cache.exists() else None\n        if row is None or row.get(\'fingerprint\')!=key:\n            if pipeline is None:pipeline=Pipeline(detector,refiner,imgsz,tta)\n            candidates=pipeline.candidates(path);methods={\'detector\':prepare_candidates(candidates,\'detector\')}\n            if refiner:\n                for threshold in (.4,.5,.6):methods[f\'refiner_{threshold}\']=prepare_candidates(candidates,\'refiner\',threshold)\n            annotations_rows=[]\n            for image_id in index.by_filename[name]:\n                gt=index.rles(image_id)\n                annotations_rows.append({\'image_id\':image_id,\'gt_count\':len(gt),\n                    \'iou\':{k:pairwise_iou(gt,[r[\'rle\'] for r in records]).tolist() for k,records in methods.items()}})\n            row={\'filename\':name,\'fingerprint\':key,\'methods\':methods,\'annotations\':annotations_rows};atomic_json(cache,row)\n        rows.append(row)\n        if (i+1)%20==0 or i+1==len(names):print(f\'Cached predictions {i+1}/{len(names)} | elapsed {(time.monotonic()-started)/60:.1f} min\',flush=True)\n    return rows\n\n\ndef image_counts(row,rule):\n    key=method_key(rule);candidates=row[\'methods\'][key]\n    keep=np.array([r[\'score\']>=rule[\'confidence\'] and r[\'area\']>=rule[\'min_area\'] for r in candidates],bool)\n    result=np.zeros(6)\n    for annotation in row[\'annotations\']:\n        iou=np.asarray(annotation[\'iou\'][key],float).reshape(annotation[\'gt_count\'],len(candidates))\n        result+=counts(iou[:,keep])\n    return result\n\n\ndef totals(rows,rule):return np.array([image_counts(row,rule) for row in rows])\n\n\ndef rule_grid(include_refiner=True):\n    methods=[(\'detector\',.5)]+([(\'refiner\',t) for t in (.4,.5,.6)] if include_refiner else [])\n    return [{\'mode\':mode,\'mask_threshold\':threshold,\'confidence\':confidence,\'min_area\':minimum}\n        for mode,threshold in methods for confidence in (.03,.08,.15,.25,.4) for minimum in (32,128)]\n\n\ndef tune(rows,include_refiner=True):\n    trials=[{**rule,**report(totals(rows,rule).sum(0))} for rule in rule_grid(include_refiner)]\n    # Fewer false positives breaks exact PQ ties; detector wins identical scores.\n    ordered=sorted(trials,key=lambda r:(-r[\'pq\'],r[\'fp\'],r[\'mode\']!=\'detector\',r[\'confidence\'],r[\'min_area\']))\n    best={k:ordered[0][k] for k in (\'mode\',\'mask_threshold\',\'confidence\',\'min_area\')}\n    return best,trials\n', 'pipeline.py': '"""Detector masks and optional instance-conditioned refinement; all outputs disjoint."""\nimport cv2\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\nfrom filament.data import encode_mask,encode_pixels,image_channels\nfrom geometry import ordered_disjoint\nfrom patches import patch_input\nfrom refiner import Refiner\n\n\nclass Pipeline:\n    def __init__(self,detector,refiner=None,imgsz=1536,tta=True):\n        self.detector=YOLO(str(detector));self.imgsz=imgsz;self.tta=tta;self.refiner=None\n        if refiner:\n            saved=torch.load(refiner,map_location=\'cpu\',weights_only=True);self.refiner=Refiner().cuda();self.refiner.load_state_dict(saved[\'model\']);self.refiner.eval();self.size=saved[\'size\']\n    @torch.inference_mode()\n    def candidates(self,path,mask_thresholds=(.4,.5,.6)):\n        result=self.detector.predict(str(path),imgsz=self.imgsz,conf=.03,iou=.5,max_det=100,retina_masks=True,device=0,verbose=False)[0]\n        records=[]\n        if result.boxes is None or result.masks is None:return records\n        boxes=result.boxes.xyxy.cpu().numpy();scores=result.boxes.conf.cpu().numpy();shape=result.orig_shape\n        channels=image_channels(path) if self.refiner is not None else None\n        for i,(box,score) in enumerate(zip(boxes,scores)):\n            raw=result.masks.data[i].cpu().numpy()>0\n            if raw.shape!=shape:raise ValueError(\'retina_masks did not return original image resolution\')\n            refined={};presence=1.\n            if self.refiner is not None:\n                patch,crop=patch_input(channels,box,self.size);x=torch.from_numpy(patch[None]).cuda();values=[];presences=[]\n                transforms=[(),(-1,),(-2,),(-2,-1)] if self.tta else [()]\n                inputs=torch.cat([x.flip(dims) if dims else x for dims in transforms],0)\n                with torch.autocast(\'cuda\',dtype=torch.float16):logits,objectness=self.refiner(inputs)\n                for j,dims in enumerate(transforms):\n                    probability=logits[j:j+1].float().sigmoid();probability=probability.flip(dims) if dims else probability\n                    values.append(probability);presences.append(objectness[j:j+1].float().sigmoid())\n                probability=torch.stack(values).mean(0)[0,0].cpu().numpy();presence=float(torch.stack(presences).mean())\n                x1,y1,x2,y2=crop;probability=cv2.resize(probability,(x2-x1,y2-y1),interpolation=cv2.INTER_LINEAR)\n                for threshold in mask_thresholds:\n                    ys,xs=np.where(probability>=threshold)\n                    refined[str(threshold)]=encode_pixels(ys+y1,xs+x1,shape)\n            records.append({\'raw\':encode_mask(raw),\'refined\':refined,\'score\':float(score),\'refined_score\':float(score)*presence})\n        return records\n\n\ndef prepare_candidates(records,mode,mask_threshold=.5,shape=(2048,2048)):\n    """Fix overlap ownership before confidence/area calibration for cacheable IoUs."""\n    from pycocotools import mask as coco\n    patches=[]\n    for r in records:\n        score=r[\'score\'] if mode==\'detector\' else r[\'refined_score\']\n        rle=r[\'raw\'] if mode==\'detector\' else r[\'refined\'][str(mask_threshold)]\n        mask=coco.decode(rle).astype(bool)\n        if mask.shape!=shape:raise ValueError(\'Unexpected mask dimensions\')\n        patches.append({\'crop\':(0,0,shape[1],shape[0]),\'mask\':mask,\'score\':score})\n    return [{\'rle\':encode_mask(r[\'mask\']),\'score\':r[\'score\'],\'area\':int(r[\'mask\'].sum())}\n            for r in ordered_disjoint(patches,shape,8)]\n\n\ndef select_prepared(records,settings):\n    return [r[\'rle\'] for r in records if r[\'score\']>=settings[\'confidence\'] and r[\'area\']>=settings[\'min_area\']]\n', 'train_detector.py': '"""Train YOLO11 segmentation, preserving an unstripped recovery checkpoint."""\nimport argparse,json,shutil,time\nfrom pathlib import Path\nimport torch\nfrom ultralytics import YOLO,settings\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--experiment\',required=True);p.add_argument(\'--weights\',default=\'yolo11m-seg.pt\')\n    p.add_argument(\'--epochs\',type=int,default=30);p.add_argument(\'--imgsz\',type=int,default=1536)\n    p.add_argument(\'--batch\',type=int,default=2);p.add_argument(\'--seed\',type=int,default=2026)\n    args=p.parse_args();out=Path(args.experiment);done=out/\'detector_complete.json\'\n    if done.exists():print(\'Detector stage already complete; reusing saved checkpoints.\');return\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a GPU.\')\n    settings.update({\'wandb\':False,\'mlflow\':False,\'comet\':False,\'clearml\':False,\'tensorboard\':False})\n    recovery=out/\'detector_resume.pt\';started=time.monotonic();epoch_times=[];timer=[started]\n    def preserve(trainer):\n        shutil.copy2(trainer.last,recovery.with_suffix(\'.tmp\'));recovery.with_suffix(\'.tmp\').replace(recovery)\n        now=time.monotonic();epoch_times.append(now-timer[0]);timer[0]=now\n        if len(epoch_times)>=2:\n            median=sorted(epoch_times[-3:])[len(epoch_times[-3:])//2]\n            print(json.dumps({\'detector_epoch\':trainer.epoch+1,\'recent_seconds_per_epoch\':median,\n                \'estimated_remaining_training_hours\':max(0,trainer.epochs-trainer.epoch-1)*median/3600}),flush=True)\n    model=YOLO(str(recovery) if recovery.exists() else args.weights);model.add_callback(\'on_model_save\',preserve)\n    if recovery.exists():\n        model.train(resume=True,data=str(out/\'detector_data.yaml\'),device=0,workers=2)\n    else:\n        model.train(data=str(out/\'detector_data.yaml\'),epochs=args.epochs,imgsz=args.imgsz,batch=args.batch,\n            device=0,workers=2,project=str(out),name=\'detector\',exist_ok=True,save_period=5,\n            optimizer=\'AdamW\',lr0=.001,lrf=.05,cos_lr=True,patience=10,seed=args.seed,\n            amp=True,cache=False,mask_ratio=2,overlap_mask=False,\n            mosaic=.15,close_mosaic=5,mixup=0.,copy_paste=0.,degrees=15.,translate=.05,scale=.15,\n            flipud=.5,fliplr=.5,hsv_h=0.,hsv_s=0.,hsv_v=.10,plots=False)\n    done.write_text(json.dumps({\'seconds\':time.monotonic()-started,\'epochs_requested\':args.epochs,\'imgsz\':args.imgsz,\'batch\':args.batch},indent=2))\n\nif __name__==\'__main__\':main()\n', 'prepare_experiment.py': '"""Build date-grouped partitions and masks-derived YOLO segmentation labels."""\nimport argparse, hashlib, json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom pycocotools import mask as coco\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom ultralytics.data.converter import merge_multi_segment\nfrom filament.data import AnnotationIndex, make_folds\nfrom filament.metrics import pairwise_iou\nfrom geometry import assert_partitions\n\n\ndef choose_annotation(index, name):\n    ids = sorted(index.by_filename[name]); candidates = [index.rles(k) for k in ids]\n    if len(ids) == 1: return ids[0]\n    scores = []\n    for i, rles in enumerate(candidates):\n        values = []\n        for j, other in enumerate(candidates):\n            if i == j: continue\n            iou = pairwise_iou(rles, other)\n            a = float(iou.max(1).sum()) if iou.shape[1] else 0.\n            b = float(iou.max(0).sum()) if iou.shape[0] else 0.\n            values.append((a+b)/max(1,len(rles)+len(other)))\n        scores.append(np.mean(values))\n    return ids[int(np.argmax(scores))]\n\n\ndef polygon_for(rle):\n    mask = coco.decode(rle).astype(np.uint8); h,w = mask.shape\n    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]\n    contours = [cv2.approxPolyDP(c,.5,True).reshape(-1,2) for c in contours]\n    contours = [c for c in contours if len(c)>=3]\n    if not contours:\n        x,y,bw,bh=map(float,coco.toBbox(rle))\n        contours=[np.array([[x,y],[x+bw,y],[x+bw,y+bh],[x,y+bh]])]\n    polygons = [c.reshape(-1).tolist() for c in contours]\n    points = contours[0] if len(contours)==1 else np.concatenate(merge_multi_segment(polygons),axis=0)\n    rendered = np.zeros_like(mask);cv2.fillPoly(rendered,[points.round().astype(np.int32)],1)\n    intersection = np.logical_and(mask,rendered).sum(); union=np.logical_or(mask,rendered).sum()\n    quality = float(intersection/max(1,union))\n    return np.clip(points/np.array([w,h]),0,1).reshape(-1),quality\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'annotations\',\'images\',\'out\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--fold\',type=int,default=0);p.add_argument(\'--seed\',type=int,default=2026)\n    args=p.parse_args();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n    index=AnnotationIndex(args.annotations); folds=pd.DataFrame(make_folds(index));folds.to_csv(out/\'folds.csv\',index=False)\n    groups=folds.set_index(\'filename\').temporal_group.to_dict()\n    validation=folds.loc[folds.fold==args.fold,\'filename\'].tolist()\n    remaining=folds.loc[folds.fold!=args.fold,\'filename\'].to_numpy()\n    train_ids,cal_ids=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=args.seed+args.fold).split(remaining,groups=[groups[n] for n in remaining]))\n    partitions={\'train\':sorted(remaining[train_ids].tolist()),\'calibration\':sorted(remaining[cal_ids].tolist()),\'validation\':sorted(validation)}\n    assert_partitions(partitions,groups)\n    manifest={\'fold\':args.fold,\'seed\':args.seed,\'partitions\':partitions,\'annotation_sha256\':hashlib.sha256(Path(args.annotations).read_bytes()).hexdigest(),\n        \'target_policy\':\'medoid annotator chosen using allowed masks only; refiner uses original RLE masks\',\n        \'test_images_used_for_training\':False}\n    manifest_path=out/\'partitions.json\'\n    if manifest_path.exists() and json.loads(manifest_path.read_text())!=manifest:raise ValueError(\'Existing experiment has different data/partitions; choose a new output directory.\')\n    manifest_path.write_text(json.dumps(manifest,indent=2)); quality=[]\n    for split in (\'train\',\'calibration\'):\n        images=out/\'detector_data\'/\'images\'/split;labels=out/\'detector_data\'/\'labels\'/split\n        images.mkdir(parents=True,exist_ok=True);labels.mkdir(parents=True,exist_ok=True)\n        for i,name in enumerate(partitions[split]):\n            image_id=choose_annotation(index,name);lines=[]\n            for j,rle in enumerate(index.rles(image_id)):\n                points,iou=polygon_for(rle);lines.append(\'0 \'+\' \'.join(f\'{v:.7f}\' for v in points))\n                quality.append({\'filename\':name,\'image_id\':image_id,\'instance\':j,\'polygon_mask_iou\':iou})\n            (labels/(Path(name).stem+\'.txt\')).write_text(\'\\n\'.join(lines)+\'\\n\')\n            link=images/name\n            if not link.exists():link.symlink_to((Path(args.images)/name).resolve())\n            if (i+1)%50==0:print(f\'Prepared {split} {i+1}/{len(partitions[split])}\',flush=True)\n    data={\'path\':str((out/\'detector_data\').resolve()),\'train\':\'images/train\',\'val\':\'images/calibration\',\'names\':{0:\'filament\'}}\n    (out/\'detector_data.yaml\').write_text(yaml.safe_dump(data))\n    pd.DataFrame(quality).to_csv(out/\'polygon_conversion.csv\',index=False)\n    print(json.dumps({\'partitions\':{k:len(v) for k,v in partitions.items()},\'polygon_quality_mean\':float(np.mean([r[\'polygon_mask_iou\'] for r in quality])),\n                     \'polygon_quality_below_0.9\':sum(r[\'polygon_mask_iou\']<.9 for r in quality)},indent=2))\n\nif __name__==\'__main__\':main()\n', 'evaluate_experiment.py': '"""Freeze calibration settings, compare held-out systems, export morphology examples."""\nimport argparse,json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom pycocotools import mask as coco\nfrom filament.data import AnnotationIndex\nfrom cached_evaluation import collect,tune,totals,method_key\nfrom metric_core import report,counts,grouped_interval\nfrom pipeline import select_prepared\n\n\ndef preview(rows,index,images,rule,out):\n    for previous in out.glob(\'preview_*.jpg\'):previous.unlink()\n    # Deterministic low-scoring and middle-scoring cases, with labels for the chosen annotator.\n    scores=[report(totals([row],rule).sum(0))[\'pq\'] for row in rows];order=np.argsort(scores)\n    chosen=list(order[:4])+list(order[len(order)//2:len(order)//2+4]);details=[]\n    for n,i in enumerate(chosen):\n        row=rows[int(i)];name=row[\'filename\'];image_id=sorted(index.by_filename[name])[0]\n        gray=cv2.imread(str(Path(images)/name),cv2.IMREAD_GRAYSCALE);gray=cv2.resize(gray,(640,640));base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)\n        panels=[]\n        pred=select_prepared(row[\'methods\'][method_key(rule)],rule)\n        for label,rles in [(\'Ground truth annotator \'+str(image_id),index.rles(image_id)),(\'Frozen prediction\',pred)]:\n            panel=base.copy()\n            for j,rle in enumerate(rles):\n                mask=cv2.resize(coco.decode(rle).astype(np.uint8),(640,640),interpolation=cv2.INTER_NEAREST)\n                contours=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]\n                color=[(0,255,255),(0,255,0),(255,150,0),(180,50,255)][j%4];cv2.drawContours(panel,contours,-1,color,1)\n            cv2.putText(panel,label,(8,25),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),1)\n            panels.append(panel)\n        canvas=np.concatenate(panels,axis=1);cv2.imwrite(str(out/f\'preview_{n+1:02d}_{Path(name).stem}.jpg\'),canvas)\n        details.append({\'filename\':name,\'pooled_image_pq\':scores[int(i)],\'displayed_annotator_image\':image_id,\'selection\':\'lowest PQ\' if n<4 else \'middle PQ\'})\n    pd.DataFrame(details).to_csv(out/\'preview_cases.csv\',index=False)\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'experiment\',\'annotations\',\'images\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--imgsz\',type=int,default=1536);p.add_argument(\'--no-tta\',action=\'store_true\')\n    args=p.parse_args();out=Path(args.experiment);index=AnnotationIndex(args.annotations)\n    partitions=json.loads((out/\'partitions.json\').read_text())[\'partitions\'];detector=out/\'detector_selected.pt\';refiner=out/\'refiner_best.pt\'\n    calibration=collect(partitions[\'calibration\'],args.images,index,args.annotations,detector,refiner,args.imgsz,out/\'calibration_cache\',not args.no_tta)\n    chosen,trials=tune(calibration,True);detector_rule,detector_trials=tune(calibration,False)\n    refinement_trials=[r for r in trials if r[\'mode\']==\'refiner\'];best_refiner=max(refinement_trials,key=lambda r:(r[\'pq\'],-r[\'fp\']))\n    refiner_rule={k:best_refiner[k] for k in (\'mode\',\'mask_threshold\',\'confidence\',\'min_area\')}\n    settings={**chosen,\'imgsz\':args.imgsz,\'tta\':not args.no_tta,\'overlap_policy\':\'all candidates above base confidence 0.03 get score-ordered disjoint ownership before final filters\',\n              \'base_min_area\':8,\'selection_partition\':\'calibration\'}\n    (out/\'selected_settings.json\').write_text(json.dumps(settings,indent=2));pd.DataFrame(trials).to_csv(out/\'calibration_trials.csv\',index=False)\n    print(\'Frozen before held-out evaluation:\',json.dumps(settings),flush=True)\n    validation=collect(partitions[\'validation\'],args.images,index,args.annotations,detector,refiner,args.imgsz,out/\'validation_cache\',not args.no_tta)\n    baseline_counts=totals(validation,detector_rule);refiner_counts=totals(validation,refiner_rule);chosen_counts=totals(validation,chosen)\n    comparisons=[{\'system\':\'detector_only\',\'selected_on_calibration\':chosen[\'mode\']==\'detector\',**report(baseline_counts.sum(0))},\n                 {\'system\':\'detector_plus_refiner\',\'selected_on_calibration\':chosen[\'mode\']==\'refiner\',**report(refiner_counts.sum(0))}]\n    pd.DataFrame(comparisons).to_csv(out/\'validation_comparison.csv\',index=False)\n    folds=pd.read_csv(out/\'folds.csv\');groups=folds.set_index(\'filename\').temporal_group.to_dict()\n    interval=grouped_interval(baseline_counts,refiner_counts,[groups[r[\'filename\']] for r in validation])\n    summary={\'fold\':json.loads((out/\'partitions.json\').read_text())[\'fold\'],\'selected_settings\':settings,\n        \'validation\':report(chosen_counts.sum(0)),\'comparisons\':comparisons,\'refiner_minus_detector_pq_group_bootstrap_95pct\':interval,\n        \'validation_images\':len(validation),\'calibration_images\':len(calibration),\n        \'interpretation\':\'Model and postprocessing settings are selected on the calibration partition, then applied unchanged to the held-out validation partition. The bootstrap interval summarizes temporal-group variability.\'}\n    (out/\'validation_summary.json\').write_text(json.dumps(summary,indent=2))\n    per_image=[];per_annotator=[];matched=[];key=method_key(chosen)\n    for row,total in zip(validation,chosen_counts):\n        per_image.append({\'filename\':row[\'filename\'],**report(total)})\n        candidates=row[\'methods\'][key];keep=np.array([r[\'score\']>=chosen[\'confidence\'] and r[\'area\']>=chosen[\'min_area\'] for r in candidates],bool)\n        for annotation in row[\'annotations\']:\n            iou=np.asarray(annotation[\'iou\'][key],float).reshape(annotation[\'gt_count\'],len(candidates))[:,keep]\n            per_annotator.append({\'filename\':row[\'filename\'],\'annotator_image\':annotation[\'image_id\'],**report(counts(iou))})\n            for value in iou[iou>.5]:matched.append({\'filename\':row[\'filename\'],\'annotator_image\':annotation[\'image_id\'],\'iou\':float(value),\'dice\':float(2*value/(1+value))})\n    pd.DataFrame(per_image).to_csv(out/\'validation_per_image.csv\',index=False)\n    pd.DataFrame(per_annotator).to_csv(out/\'validation_per_annotator.csv\',index=False)\n    pd.DataFrame(matched,columns=[\'filename\',\'annotator_image\',\'iou\',\'dice\']).to_csv(out/\'validation_matched_overlaps.csv\',index=False)\n    preview(validation,index,args.images,chosen,out)\n    print(json.dumps(summary,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'train_refiner.py': '"""Balanced positive/negative patch training with epoch recovery checkpoints."""\nimport argparse,json,time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader,Dataset,WeightedRandomSampler\nfrom refiner import Refiner,objective\n\n\nclass Patches(Dataset):\n    def __init__(self,rows,augment=False,seed=2026):self.rows=rows;self.augment=augment;self.epoch=0;self.seed=seed\n    def __len__(self):return len(self.rows)\n    def __getitem__(self,i):\n        row=self.rows[i]\n        with np.load(row[\'path\']) as data:x=data[\'x\'].astype(np.float32)/255.;y=data[\'target\'][None].astype(np.float32);valid=data[\'valid\'][None].astype(np.float32)\n        if self.augment:\n            rng=np.random.default_rng(self.seed+self.epoch*100000+i);k=int(rng.integers(4))\n            x,y,valid=[np.rot90(a,k,axes=(-2,-1)) for a in (x,y,valid)]\n            if rng.random()<.5:x,y,valid=[a[...,::-1] for a in (x,y,valid)]\n            x[:2]=np.clip(x[:2]*rng.uniform(.85,1.15),0,1)\n        return tuple(torch.from_numpy(np.ascontiguousarray(a)) for a in (x,y,valid))+(torch.tensor(float(row[\'positive\'])),)\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--experiment\',required=True);p.add_argument(\'--encoder-weights\')\n    p.add_argument(\'--epochs\',type=int,default=15);p.add_argument(\'--steps\',type=int,default=150);p.add_argument(\'--batch\',type=int,default=8)\n    p.add_argument(\'--seed\',type=int,default=2026);args=p.parse_args();out=Path(args.experiment)\n    if (out/\'refiner_complete.json\').exists():print(\'Refiner stage complete; reusing saved model.\');return\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a GPU.\')\n    torch.set_num_threads(2);torch.manual_seed(args.seed)\n    payload=json.loads((out/\'patches.json\').read_text());train=[r for r in payload[\'records\'] if r[\'split\']==\'train\'];val=[r for r in payload[\'records\'] if r[\'split\']==\'calibration\']\n    npos=sum(r[\'positive\'] for r in train);nneg=len(train)-npos\n    if not npos or not nneg or not val:raise ValueError(\'Need positive, negative and calibration patches.\')\n    last=out/\'refiner_last.pt\';model=Refiner(pretrained=not last.exists(),encoder_weights=args.encoder_weights if not last.exists() else None).cuda()\n    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,args.epochs,eta_min=1e-5)\n    scaler=torch.amp.GradScaler(\'cuda\');start=0;best=float(\'inf\');bad=0;history=[]\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=True);model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);scheduler.load_state_dict(saved[\'scheduler\']);scaler.load_state_dict(saved[\'scaler\'])\n        start=saved[\'epoch\'];best=saved[\'best\'];bad=saved[\'bad\'];history=saved[\'history\']\n    weights=[.75/npos if r[\'positive\'] else .25/nneg for r in train]\n    ds=Patches(train,True,args.seed);vd=Patches(val,False,args.seed)\n    vl=DataLoader(vd,batch_size=args.batch,shuffle=False,num_workers=2,pin_memory=True)\n    for epoch in range(start,args.epochs):\n        began=time.monotonic();ds.epoch=epoch;generator=torch.Generator().manual_seed(args.seed+epoch)\n        sampler=WeightedRandomSampler(weights,args.steps*args.batch,replacement=True,generator=generator)\n        loader=DataLoader(ds,batch_size=args.batch,sampler=sampler,num_workers=2,pin_memory=True)\n        model.train();train_loss=0.\n        for x,y,valid,positive in loader:\n            x,y,valid,positive=[a.cuda(non_blocking=True) for a in (x,y,valid,positive)];optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):mask,presence=model(x);loss=objective(mask,presence,y,valid,positive)\n            if not torch.isfinite(loss):raise RuntimeError(\'Non-finite loss; recovery checkpoint remains at previous epoch.\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.)\n            scaler.step(optimizer);scaler.update();train_loss+=float(loss.detach())\n        model.eval();val_loss=0.;n=0\n        with torch.inference_mode():\n            for x,y,valid,positive in vl:\n                x,y,valid,positive=[a.cuda(non_blocking=True) for a in (x,y,valid,positive)]\n                with torch.autocast(\'cuda\',dtype=torch.float16):mask,presence=model(x);loss=objective(mask,presence,y,valid,positive)\n                val_loss+=float(loss)*len(x);n+=len(x)\n        val_loss/=max(1,n);scheduler.step();improved=val_loss<best-1e-5\n        if improved:best=val_loss;bad=0\n        else:bad+=1\n        row={\'epoch\':epoch+1,\'train_loss\':train_loss/args.steps,\'calibration_patch_loss\':val_loss,\'seconds\':time.monotonic()-began};history.append(row)\n        print(json.dumps({**row,\'estimated_remaining_refiner_hours\':row[\'seconds\']*max(0,args.epochs-epoch-1)/3600}),flush=True)\n        checkpoint={\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'scheduler\':scheduler.state_dict(),\'scaler\':scaler.state_dict(),\n            \'epoch\':epoch+1,\'best\':best,\'bad\':bad,\'history\':history,\'size\':payload[\'size\'],\'config\':vars(args)}\n        torch.save(checkpoint,last.with_suffix(\'.tmp\'));last.with_suffix(\'.tmp\').replace(last)\n        if improved:torch.save({\'model\':model.state_dict(),\'size\':payload[\'size\'],\'epoch\':epoch+1},out/\'refiner_best.pt\')\n        (out/\'refiner_history.json\').write_text(json.dumps(history,indent=2))\n        if bad>=5:print(\'Refiner stopped after five calibration-loss checks without improvement.\',flush=True);break\n    (out/\'refiner_complete.json\').write_text(json.dumps({\'epochs_finished\':len(history),\'best_calibration_patch_loss\':best},indent=2))\n\nif __name__==\'__main__\':main()\n', 'select_detector.py': '"""Select among saved detector checkpoints by pooled PQ on calibration only."""\nimport argparse,json,shutil\nfrom pathlib import Path\nimport pandas as pd\nfrom filament.data import AnnotationIndex\nfrom cached_evaluation import collect,tune,sha\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'experiment\',\'images\',\'annotations\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--imgsz\',type=int,default=1536);args=p.parse_args();out=Path(args.experiment)\n    if (out/\'detector_selection.json\').exists() and (out/\'detector_selected.pt\').exists():print(\'Detector PQ selection already complete.\');return\n    names=json.loads((out/\'partitions.json\').read_text())[\'partitions\'][\'calibration\'];index=AnnotationIndex(args.annotations)\n    candidates=sorted((out/\'detector\'/\'weights\').glob(\'epoch*.pt\'))\n    candidates += [out/\'detector\'/\'weights\'/\'best.pt\',out/\'detector\'/\'weights\'/\'last.pt\']\n    candidates=[p for p in candidates if p.is_file()];seen=set();trials=[];selected=None;best=-1.\n    for checkpoint in candidates:\n        digest=sha(checkpoint)\n        if digest in seen:continue\n        seen.add(digest);print(\'Calibration PQ for\',checkpoint.name,flush=True)\n        rows=collect(names,args.images,index,args.annotations,checkpoint,None,args.imgsz,out/\'detector_selection_cache\'/digest[:12],tta=False)\n        rule,table=tune(rows,False);score=max(r[\'pq\'] for r in table)\n        trials.append({\'checkpoint\':str(checkpoint),\'pq\':score,\'rule\':rule})\n        if score>best:best=score;selected=checkpoint\n    if selected is None:raise RuntimeError(\'No detector checkpoint available.\')\n    shutil.copy2(selected,out/\'detector_selected.pt\')\n    (out/\'detector_selection.json\').write_text(json.dumps({\'selected\':str(selected),\'calibration_pq\':best,\'trials\':trials},indent=2))\n    print(\'Selected by calibration PQ:\',selected,\'PQ:\',best,flush=True)\n\nif __name__==\'__main__\':main()\n', 'patches.py': '"""Use original masks for refiner targets and dark/false-proposal background patches."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as coco\nfrom ultralytics import YOLO\nfrom filament.data import AnnotationIndex,image_channels\nfrom geometry import context_crop,box_iou\n\n\ndef patch_input(channels,box,size=256):\n    _,h,w=channels.shape;crop=context_crop(box,w,h);x1,y1,x2,y2=crop\n    gray=cv2.resize(channels[0,y1:y2,x1:x2],(size,size),interpolation=cv2.INTER_AREA)\n    contrast=cv2.resize(channels[1,y1:y2,x1:x2],(size,size),interpolation=cv2.INTER_AREA)\n    seed=np.zeros((y2-y1,x2-x1),np.uint8)\n    xa=max(0,int(np.floor(box[0]))-x1);ya=max(0,int(np.floor(box[1]))-y1)\n    xb=min(seed.shape[1],int(np.ceil(box[2]))-x1);yb=min(seed.shape[0],int(np.ceil(box[3]))-y1)\n    if xb>xa and yb>ya:seed[ya:yb,xa:xb]=1\n    seed=cv2.resize(seed,(size,size),interpolation=cv2.INTER_NEAREST)\n    return np.stack([gray,contrast,seed]).astype(np.float32),crop\n\n\ndef save_patch(path,channels,box,truth,union,size):\n    x,crop=patch_input(channels,box,size);x1,y1,x2,y2=crop\n    target=cv2.resize(truth[y1:y2,x1:x2],(size,size),interpolation=cv2.INTER_NEAREST)\n    occupied=cv2.resize(union[y1:y2,x1:x2],(size,size),interpolation=cv2.INTER_NEAREST)\n    valid=((occupied==0)|(target>0)).astype(np.uint8)\n    np.savez_compressed(path,x=np.round(x*255).astype(np.uint8),target=target,valid=valid)\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'experiment\',\'annotations\',\'images\',\'detector\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--size\',type=int,default=256);p.add_argument(\'--imgsz\',type=int,default=1536)\n    args=p.parse_args();out=Path(args.experiment);root=out/\'patches\';root.mkdir(exist_ok=True)\n    descriptor={\'detector_sha256\':hashlib.sha256(Path(args.detector).read_bytes()).hexdigest(),\'size\':args.size,\'imgsz\':args.imgsz,\n        \'annotations_sha256\':hashlib.sha256(Path(args.annotations).read_bytes()).hexdigest(),\'partitions_sha256\':hashlib.sha256((out/\'partitions.json\').read_bytes()).hexdigest()}\n    if (out/\'patches.json\').exists():\n        saved=json.loads((out/\'patches.json\').read_text())\n        if saved.get(\'descriptor\')!=descriptor:raise ValueError(\'Patch cache belongs to a different model or experiment.\')\n        if all(Path(r[\'path\']).is_file() for r in saved[\'records\']):print(\'Patch generation already complete; using cached targets.\');return\n    progress=out/\'patch_generation.json\'\n    if progress.exists() and json.loads(progress.read_text())!=descriptor:raise ValueError(\'Partial patch cache belongs to a different experiment.\')\n    progress.write_text(json.dumps(descriptor))\n    partitions=json.loads((out/\'partitions.json\').read_text())[\'partitions\'];index=AnnotationIndex(args.annotations)\n    detector=YOLO(args.detector);rows=[];rng=np.random.default_rng(2026)\n    for split in (\'train\',\'calibration\'):\n        for i,name in enumerate(partitions[split]):\n            channels=image_channels(Path(args.images)/name);_,h,w=channels.shape\n            rles=[r for k in index.by_filename[name] for r in index.rles(k)]\n            union=np.zeros((h,w),np.uint8);boxes=[]\n            for r in rles:\n                union|=coco.decode(r).astype(np.uint8);x,y,bw,bh=map(float,coco.toBbox(r));boxes.append([x,y,x+bw,y+bh])\n            for j,(rle,box) in enumerate(zip(rles,boxes)):\n                key=f\'{split}_{Path(name).stem}_positive_{j:03d}.npz\';path=root/key\n                if not path.exists():save_patch(path,channels,box,coco.decode(rle).astype(np.uint8),union,args.size)\n                rows.append({\'path\':str(path.resolve()),\'split\':split,\'positive\':1,\'filename\':name,\'kind\':\'mask_target\'})\n            negatives=[]\n            if split==\'train\':\n                result=detector.predict(str(Path(args.images)/name),imgsz=args.imgsz,conf=.08,iou=.5,max_det=100,retina_masks=True,device=0,verbose=False)[0]\n                if result.boxes is not None:\n                    for box in result.boxes.xyxy.cpu().numpy():\n                        xa,ya,xb,yb=np.round(box).astype(int);xa,ya=max(0,xa),max(0,ya);xb,yb=min(w,xb),min(h,yb)\n                        if xb>xa and yb>ya and union[ya:yb,xa:xb].sum()==0 and (not boxes or box_iou(box,boxes).max()<.1):\n                            negatives.append((box.tolist(),\'detector_false_proposal\'))\n            dark=(channels[2]>.15)&(channels[0]>.05)&(union==0);ys,xs=np.where(dark)\n            for attempt in range(60):\n                if len(negatives)>=6 or not len(xs):break\n                j=int(rng.integers(len(xs)));side=int(rng.integers(40,160));cx,cy=int(xs[j]),int(ys[j])\n                box=[max(0,cx-side//2),max(0,cy-side//2),min(w,cx+side//2),min(h,cy+side//2)]\n                xa,ya,xb,yb=box\n                if xb>xa and yb>ya and union[ya:yb,xa:xb].sum()==0:negatives.append((box,\'dark_background\'))\n            for j,(box,kind) in enumerate(negatives[:10]):\n                key=f\'{split}_{Path(name).stem}_negative_{j:03d}.npz\';path=root/key\n                if not path.exists():save_patch(path,channels,box,np.zeros((h,w),np.uint8),union,args.size)\n                rows.append({\'path\':str(path.resolve()),\'split\':split,\'positive\':0,\'filename\':name,\'kind\':kind})\n            if (i+1)%25==0:print(f\'Refiner patches {split} {i+1}/{len(partitions[split])}\',flush=True)\n    if not any(r[\'positive\']==0 and r[\'split\']==\'train\' for r in rows):raise RuntimeError(\'No valid negative training patches.\')\n    payload={\'size\':args.size,\'records\':rows,\'descriptor\':descriptor}\n    (out/\'patches.json\').write_text(json.dumps(payload))\n    print(json.dumps({\'patches\':len(rows),\'negative_training_patches\':sum(r[\'split\']==\'train\' and not r[\'positive\'] for r in rows)},indent=2))\n\nif __name__==\'__main__\':main()\n', 'filament/__init__.py': '"""Mask-only MAGFiLO segmentation research pipeline."""\n__version__ = "0.1.0"\n', 'filament/metrics.py': '"""Organizer v6 semantics: all qualifying pairs, per-annotator comparison, pooled counts."""\nfrom dataclasses import dataclass,asdict\nimport numpy as np\nfrom pycocotools import mask as coco\n\n@dataclass\nclass Counts:\n    tp:int=0\n    fp:int=0\n    fn:int=0\n    sum_iou:float=0.\n    split_gt:int=0\n    merged_predictions:int=0\n    def add(self,other):\n        for key in asdict(self): setattr(self,key,getattr(self,key)+getattr(other,key))\n        return self\n    def report(self):\n        d=self.tp+.5*self.fp+.5*self.fn\n        return {**asdict(self),"pq":self.sum_iou/d if d else 0.,"sq":self.sum_iou/self.tp if self.tp else 0.,"rq":self.tp/d if d else 0.}\n\ndef overlap_counts(iou):\n    iou=np.asarray(iou,np.float64);hit=iou>.5;overlap=iou>0\n    return Counts(int(hit.sum()),int((hit.sum(0)==0).sum()),int((hit.sum(1)==0).sum()),float(iou[hit].sum()),int((overlap.sum(1)>1).sum()),int((overlap.sum(0)>1).sum()))\n\ndef pairwise_iou(gt,pred):\n    if not gt or not pred:return np.zeros((len(gt),len(pred)),np.float64)\n    return np.asarray(coco.iou(pred,gt,[0]*len(gt)),np.float64).T\n\ndef evaluate_image(index,filename,predicted_rles):\n    total,rows=Counts(),[]\n    for image_id in index.by_filename[filename]:\n        count=overlap_counts(pairwise_iou(index.rles(image_id),predicted_rles));total.add(count)\n        rows.append({"filename":filename,"annotator_image":image_id,**count.report()})\n    return total,rows\n', 'filament/data.py': 'from __future__ import annotations\nimport json\nfrom collections import defaultdict\nfrom datetime import datetime\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as coco\nfrom sklearn.model_selection import GroupKFold\n\n\ndef annotation_rle(annotation, height=2048, width=2048):\n    seg = annotation["segmentation"]\n    if isinstance(seg, list):\n        if not seg: raise ValueError(f"Empty polygon: {annotation[\'id\']}")\n        return coco.merge(coco.frPyObjects(seg, height, width))\n    if isinstance(seg["counts"], list): return coco.frPyObjects(seg, height, width)\n    return {"size": list(seg["size"]), "counts": seg["counts"].encode() if isinstance(seg["counts"], str) else seg["counts"]}\n\n\ndef encode_mask(mask):\n    rle = coco.encode(np.asfortranarray(mask.astype(np.uint8)))\n    return {"size": list(rle["size"]), "counts": rle["counts"].decode("ascii")}\n\n\ndef encode_pixels(ys, xs, shape=(2048, 2048)):\n    height, width = shape\n    positions = np.unique(np.asarray(xs, dtype=np.int64)*height + np.asarray(ys, dtype=np.int64))\n    if not len(positions): counts = [height*width]\n    else:\n        cuts = np.where(np.diff(positions) != 1)[0]\n        starts = positions[np.r_[0, cuts+1]]\n        ends = positions[np.r_[cuts, len(positions)-1]]+1\n        counts = np.column_stack([starts-np.r_[0,ends[:-1]], ends-starts]).ravel().tolist()\n        counts.append(int(height*width-ends[-1]))\n    rle = coco.frPyObjects({"size":[height,width], "counts":counts},height,width)\n    return {"size":[height,width], "counts":rle["counts"].decode("ascii")}\n\n\nclass AnnotationIndex:\n    def __init__(self, annotation_path):\n        data=json.loads(Path(annotation_path).read_text())\n        self.images={str(i["id"]):i for i in data["images"]}\n        self.annotations=defaultdict(list)\n        for a in data["annotations"]:\n            if str(a["image_id"]) not in self.images: raise ValueError("Missing image record")\n            self.annotations[str(a["image_id"])].append(a)\n        self.by_filename=defaultdict(list)\n        for image_id, image in self.images.items(): self.by_filename[Path(image["file_name"]).name].append(image_id)\n        self.filenames=sorted(self.by_filename)\n\n    def rles(self, image_id):\n        image=self.images[image_id]\n        return [annotation_rle(a,image["height"],image["width"]) for a in self.annotations[image_id]]\n\n    def labels(self,image_id):\n        image=self.images[image_id]\n        labels=np.zeros((image["height"],image["width"]),np.int32)\n        for k,rle in enumerate(self.rles(image_id),1):\n            mask=coco.decode(rle).astype(bool); overlap=mask&(labels!=0)\n            labels[mask&(labels==0)]=k; labels[overlap]=-1\n        return labels\n\n\ndef make_folds(index,n_splits=5,gap_days=3):\n    names=np.array(index.filenames); mapping={}; group=-1; previous=None\n    for date,name in sorted((datetime.strptime(Path(n).stem[:8],"%Y%m%d"),n) for n in names):\n        if previous is None or (date-previous).days>gap_days: group+=1\n        mapping[name]=group; previous=date\n    groups=np.array([mapping[n] for n in names])\n    if len(set(groups))<n_splits: raise ValueError("Too few independent temporal groups")\n    result=[]\n    for fold,(_,val) in enumerate(GroupKFold(n_splits).split(names,groups=groups)):\n        for n in names[val]: result.append({"filename":str(n),"fold":fold,"temporal_group":mapping[n],"station":Path(n).stem[-2:],"date":str(n)[:8]})\n    return sorted(result,key=lambda r:r["filename"])\n\n\ndef image_channels(path):\n    image=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)\n    if image is None: raise ValueError(f"Cannot read {path}")\n    contrast=cv2.createCLAHE(clipLimit=1.5,tileGridSize=(8,8)).apply(image)\n    contrast[image==0]=0\n    # Counterfactual channel: estimate how much darker each pixel is than its\n    # local surroundings at two solar-feature scales.  Filaments should remain\n    # dark in both the raw image and this explicit local-contrast view, while\n    # many limb/texture artefacts do not.\n    small=cv2.GaussianBlur(image,(0,0),9)\n    large=cv2.GaussianBlur(image,(0,0),33)\n    darkness=np.maximum(small.astype(np.float32)-image,large.astype(np.float32)-image)\n    darkness=np.clip(darkness,0,64)*(255./64.)\n    darkness[image==0]=0\n    return np.stack([image,contrast,darkness]).astype(np.float32)/255.\n\n\ndef slice_pair(shape,offset):\n    h,w=shape;dy,dx=offset\n    if abs(dy)>=h or abs(dx)>=w: return (slice(0,0),slice(0,0)),(slice(0,0),slice(0,0))\n    return (slice(max(0,-dy),min(h,h-dy)),slice(max(0,-dx),min(w,w-dx))), (slice(max(0,dy),min(h,h+dy)),slice(max(0,dx),min(w,w+dx)))\n\n\ndef affinity_targets(labels,offsets):\n    targets=np.zeros((len(offsets),*labels.shape),np.float32);valid=np.zeros_like(targets)\n    for k,offset in enumerate(offsets):\n        a,b=slice_pair(labels.shape,offset); left,right=labels[a],labels[b]\n        usable=(left>0)&(right>0)\n        targets[k][a]=(left==right)&usable;valid[k][a]=usable\n    return targets,valid\n'}
for name, text in SOURCES.items():
    target = PROJECT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text)

def run(*arguments):
    subprocess.run([sys.executable, *map(str, arguments)], cwd=PROJECT, check=True)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

# Optional: attach the recovery archive to continue an interrupted run.
restore = list(Path('/kaggle/input').rglob('filament_v5_results.zip'))
if restore:
    if len({digest(p) for p in restore}) != 1: raise ValueError('Different recovery archives found.')
    with zipfile.ZipFile(restore[0]) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            if member.is_dir(): continue
            if relative.is_absolute() or '..' in relative.parts: raise ValueError('Unsafe recovery archive path')
            if relative.parts[0] not in {'experiments', 'pretrained'}: continue
            target = PROJECT / relative
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as destination: shutil.copyfileobj(source, destination)
    print('Restored saved checkpoints and reports.')

# Settings: changing training settings creates a separate experiment directory.
FOLDS = [0]
DETECTOR_MODEL = 'yolo11m-seg.pt'
DETECTOR_EPOCHS = 30
DETECTOR_SIZE = 1536
DETECTOR_BATCH = 2
REFINER_EPOCHS = 15
REFINER_STEPS = 150
REFINER_BATCH = 8
REFINER_SIZE = 384
SEED = 2026
RUN_TAG = 'run1'
TRAINING = dict(detector_model=DETECTOR_MODEL, detector_epochs=DETECTOR_EPOCHS,
                detector_size=DETECTOR_SIZE, detector_batch=DETECTOR_BATCH,
                refiner_epochs=REFINER_EPOCHS, refiner_steps=REFINER_STEPS,
                refiner_batch=REFINER_BATCH, refiner_size=REFINER_SIZE, seed=SEED,
                run_tag=RUN_TAG, pipeline_version='detref-v5-1')
config_id = hashlib.sha256(json.dumps(TRAINING, sort_keys=True).encode()).hexdigest()[:10]
EXPERIMENTS = {fold: PROJECT / 'experiments' / f'fold{fold}_{config_id}' for fold in FOLDS}
for fold, experiment in EXPERIMENTS.items():
    experiment.mkdir(parents=True, exist_ok=True)
    (experiment / 'training_config.json').write_text(json.dumps(TRAINING, indent=2))
print('Experiments:', EXPERIMENTS)

# Actual runtime versions are preserved for later reproduction.
with (PROJECT / 'requirements-observed.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)


In [ ]:
import torch, torchvision, cv2, ultralytics
from pycocotools import mask as coco
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__, '| torchvision:', torchvision.__version__)
assert len(FOLDS) == len(set(FOLDS)) and all(f in range(5) for f in FOLDS)
DATA = Path('/kaggle/working/filament_data_v2')
ANNOTATIONS = DATA / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
if not ANNOTATIONS.is_file() or len(list((DATA / 'train_images').glob('*'))) != 707 or len(list((DATA / 'test_images').glob('*'))) != 180:
    DATA = Path('/kaggle/working/filament_v5_data')
    run('prepare_data.py', '--input', '/kaggle/input', '--output', DATA,
        '--folds', PROJECT / 'all_folds.csv')
    ANNOTATIONS = DATA / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TRAIN_IMAGES, TEST_IMAGES = DATA / 'train_images', DATA / 'test_images'
assert len(list(TRAIN_IMAGES.glob('*'))) == 707
assert len(list(TEST_IMAGES.glob('*'))) == 180
assert not {p.name for p in TRAIN_IMAGES.iterdir()} & {p.name for p in TEST_IMAGES.iterdir()}
for fold, experiment in EXPERIMENTS.items():
    run('prepare_experiment.py', '--annotations', ANNOTATIONS, '--images', TRAIN_IMAGES,
        '--out', experiment, '--fold', fold, '--seed', SEED)


In [ ]:
PRETRAINED = PROJECT / 'pretrained'
PRETRAINED.mkdir(exist_ok=True)
DETECTOR_WEIGHTS = PRETRAINED / DETECTOR_MODEL
ENCODER_WEIGHTS = PRETRAINED / 'resnet18-f37072fd.pth'
for target in (DETECTOR_WEIGHTS, ENCODER_WEIGHTS):
    if not target.exists():
        candidates = list(Path('/kaggle/input').rglob(target.name))
        if len({digest(p) for p in candidates}) > 1: raise ValueError('Conflicting pretrained weights: ' + target.name)
        if candidates: shutil.copy2(candidates[0], target)
if not ENCODER_WEIGHTS.exists():
    torch.hub.download_url_to_file('https://download.pytorch.org/models/resnet18-f37072fd.pth',
                                  str(ENCODER_WEIGHTS), hash_prefix='f37072fd')
assert digest(ENCODER_WEIGHTS).startswith('f37072fd'), 'Encoder weights checksum mismatch'
first_experiment = EXPERIMENTS[FOLDS[0]]
first_train = json.loads((first_experiment / 'partitions.json').read_text())['partitions']['train'][0]
run('smoke_test.py', '--detector-weights', DETECTOR_WEIGHTS,
    '--encoder-weights', ENCODER_WEIGHTS, '--image', TRAIN_IMAGES / first_train, '--size', REFINER_SIZE)
(PROJECT / 'pretrained_manifest.json').write_text(json.dumps(
    {p.name: digest(p) for p in (DETECTOR_WEIGHTS, ENCODER_WEIGHTS)}, indent=2))


## Checkpoints and recovery

Training writes checkpoints at epoch boundaries. The recovery archive stores the code, model weights, partitions, settings, prediction caches and reports. Training images and patch arrays are excluded; the official dataset is required to regenerate them in a restored session.

Download `filament_v5_results.zip` after a completed stage before stopping the session. Attach the archive as an input to resume an interrupted run. Recovery continues with the saved training settings. Changing the configuration creates a separate experiment directory.


In [ ]:
RESULTS_ARCHIVE = Path('/kaggle/working/filament_v5_results.zip')
def save_progress():
    temporary = RESULTS_ARCHIVE.with_suffix('.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in PROJECT.rglob('*'):
            if not path.is_file() or path.is_symlink(): continue
            relative = path.relative_to(PROJECT)
            if any(part in {'detector_data', 'patches', '__pycache__'} for part in relative.parts): continue
            if path.suffix == '.tmp': continue
            # After detector PQ selection, keep selected/recovery weights, not every earlier snapshot.
            if 'detector' in relative.parts and 'weights' in relative.parts:
                experiment = PROJECT / Path(*relative.parts[:2])
                if (experiment / 'detector_selected.pt').exists(): continue
            archive.write(path, relative.as_posix())
    temporary.replace(RESULTS_ARCHIVE)
    current = Path.cwd()
    try:
        os.chdir('/kaggle/working')
        display(FileLink(RESULTS_ARCHIVE.name))
    finally: os.chdir(current)

def stage(*arguments):
    try: run(*arguments)
    finally: save_progress()


## 1. Train the instance detector

For images with multiple annotation sets, the detector uses the set with the greatest mask agreement with the other annotators. Training polygons and bounding boxes are derived from these segmentation masks. The conversion report records the overlap between each original mask and its polygon representation.

The detector processes full-disk images at 1536 pixels. Saved checkpoints are evaluated using pooled Panoptic Quality (PQ) on the calibration images. The checkpoint with the highest calibration PQ is selected for the next stage. The held-out validation images are reserved for the final evaluation.

**Outputs:** detector checkpoints, a polygon conversion report, calibration scores and the selected detector weights.


In [ ]:
for fold, experiment in EXPERIMENTS.items():
    # Save after both training and PQ selection so the archive includes the selected detector.
    try:
        run('train_detector.py', '--experiment', experiment, '--weights', DETECTOR_WEIGHTS,
            '--epochs', DETECTOR_EPOCHS, '--imgsz', DETECTOR_SIZE,
            '--batch', DETECTOR_BATCH, '--seed', SEED + fold)
        run('select_detector.py', '--experiment', experiment, '--images', TRAIN_IMAGES,
            '--annotations', ANNOTATIONS, '--imgsz', DETECTOR_SIZE)
    finally: save_progress()


## 2. Prepare instance crops and background examples

Positive examples use the original RLE masks from the training annotations. Each crop contains grayscale intensity, CLAHE contrast and a seed box identifying the target instance. Other annotated filaments inside the crop are excluded from the pixel loss where they are not part of the target mask.

Background seed boxes contain no annotated filament across the available annotators. Dark image regions supply background examples, and false detector proposals are mined from the training partition. During refiner training, the sampler selects 75% positive examples and 25% background examples.

**Outputs:** cached crop arrays, target masks, valid-pixel masks and a patch manifest.


In [ ]:
for fold, experiment in EXPERIMENTS.items():
    stage('patches.py', '--experiment', experiment, '--annotations', ANNOTATIONS,
          '--images', TRAIN_IMAGES, '--detector', experiment / 'detector_selected.pt',
          '--size', REFINER_SIZE, '--imgsz', DETECTOR_SIZE)


## 3. Train the mask refiner

A pretrained ResNet18 encoder and a GroupNorm decoder predict an instance mask and a foreground-presence score. The mask loss combines boundary-weighted binary cross-entropy with Dice loss for positive examples. A separate presence loss teaches the model to distinguish filament candidates from background.

The encoder's BatchNorm running statistics remain fixed during training. Calibration patch loss selects the refiner checkpoint, with early stopping after five evaluations without improvement. Epoch checkpoints retain the model, optimizer and scheduler states for recovery.

**Outputs:** selected refiner weights, a recovery checkpoint and training history.


In [ ]:
for fold, experiment in EXPERIMENTS.items():
    stage('train_refiner.py', '--experiment', experiment, '--encoder-weights', ENCODER_WEIGHTS,
          '--epochs', REFINER_EPOCHS, '--steps', REFINER_STEPS,
          '--batch', REFINER_BATCH, '--seed', SEED + fold)


## 4. Calibrate and evaluate the pipeline

The refiner averages predictions from four flip views. Calibration compares the detector masks with the refined masks and selects the output method, confidence threshold, minimum mask area and refiner mask threshold. Overlapping candidates receive disjoint pixel ownership in descending score order before the final filters are applied.

The selected settings are frozen and applied to the held-out validation images. Evaluation reports pooled PQ, segmentation quality (SQ), recognition quality (RQ), true positives, false positives and false negatives. Additional reports summarize fragmentation, merging and the Dice/IoU values of matched masks. A paired bootstrap over temporal groups compares the detector and refiner outputs.

Segmentation previews show low-scoring and middle-scoring validation cases. Each preview identifies the displayed annotator and places the ground-truth outlines beside the predicted outlines.

**Outputs:** frozen prediction settings, calibration trials, validation metrics, mask-overlap reports and segmentation previews.


In [ ]:
import pandas as pd
summaries = []
for fold, experiment in EXPERIMENTS.items():
    stage('evaluate_experiment.py', '--experiment', experiment, '--images', TRAIN_IMAGES,
          '--annotations', ANNOTATIONS, '--imgsz', DETECTOR_SIZE)
    summary = json.loads((experiment / 'validation_summary.json').read_text())
    summaries.append({'fold': fold, **summary['validation']})
    display(pd.read_csv(experiment / 'validation_comparison.csv'))
    print('Frozen settings:', summary['selected_settings'])
display(pd.DataFrame(summaries))
(PROJECT / 'fold_summary.csv').write_text(pd.DataFrame(summaries).to_csv(index=False))
if len(summaries) == 5:
    print('Mean fold PQ:', sum(s['pq'] for s in summaries) / 5)
    total = {key: sum(s[key] for s in summaries) for key in ('tp', 'fp', 'fn', 'sum_iou')}
    print('Pooled OOF PQ:', total['sum_iou'] / (total['tp'] + .5*total['fp'] + .5*total['fn']))


In [ ]:
from IPython.display import Image
preview_experiment = EXPERIMENTS[FOLDS[0]]
display(pd.read_csv(preview_experiment / 'preview_cases.csv'))
for image in sorted(preview_experiment.glob('preview_*.jpg'))[:4]:
    display(Image(filename=str(image), width=1000))


## 5. Generate the test submission

The prediction stage processes all 180 test images using the trained weights and frozen calibration settings. The training annotation JSON is used during fitting and evaluation; the test prediction command accepts images and model settings.

Each predicted instance is encoded as compressed COCO RLE at 2048 × 2048 pixels. The CSV contains one row per detected filament with the columns `filament_id` and `segmentation_rle`. The validation check decodes every submitted mask and checks image identifiers, unique instance identifiers, nonempty masks and disjoint predictions.

`TEST_FOLD` selects the model used for test inference. When multiple folds are trained, their validation summaries are reported separately. This notebook exports predictions from the configured fold.

**Outputs:** `submission_v5.csv`, a submission-check report and the results archive. Upload the CSV through the competition's submission interface.


In [ ]:
TEST_FOLD = FOLDS[0]
TEST_EXPERIMENT = EXPERIMENTS[TEST_FOLD]
SUBMISSION = PROJECT / 'submission_v5.csv'
stage('predict_test.py', '--experiment', TEST_EXPERIMENT,
      '--images', TEST_IMAGES, '--out', SUBMISSION)
run('validate_submission.py', '--submission', SUBMISSION, '--images', TEST_IMAGES)
submission = pd.read_csv(SUBMISSION)
print('Rows:', len(submission), '| Unique filament IDs:', submission.filament_id.nunique())
display(submission.head())
save_progress()
current = Path.cwd()
try:
    os.chdir('/kaggle/working')
    display(FileLink('filament_project_v5/submission_v5.csv'))
    display(FileLink(RESULTS_ARCHIVE.name))
finally: os.chdir(current)
print('Submission CSV and validation reports are ready for review. Upload submission_v5.csv to the competition.')


## References and reproducibility

The notebook records the runtime package versions, data partitions, pretrained-weight checksums, selected settings and model checkpoints. Prediction caches retain the input, model, source-code and runtime fingerprints used to produce each result.

- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/)
- [Ultralytics segmentation dataset format](https://docs.ultralytics.com/datasets/segment/)
- [Ultralytics model training and checkpoint recovery](https://docs.ultralytics.com/modes/train/)
- [Torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [Detector and mask-refiner workflow study](https://github.com/HeShen-1/filament-segmentation-2026)

Ultralytics and Torchvision provide the pretrained models and library components. Preserve their software and model attribution when distributing the complete pipeline. Ultralytics provides its code and models under AGPL-3.0 and Enterprise licensing options.
